In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession
        .builder
        .appName("Fraud Detection AI")
        .getOrCreate()
)

In [3]:
print(spark.version)

4.2.0


In [3]:
spark


In [7]:
df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("../data/raw/creditcard.csv")
)

In [4]:
df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [7]:
df.columns

['Time',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount',
 'Class']

In [9]:
df.dtypes

[('Time', 'double'),
 ('V1', 'double'),
 ('V2', 'double'),
 ('V3', 'double'),
 ('V4', 'double'),
 ('V5', 'double'),
 ('V6', 'double'),
 ('V7', 'double'),
 ('V8', 'double'),
 ('V9', 'double'),
 ('V10', 'double'),
 ('V11', 'double'),
 ('V12', 'double'),
 ('V13', 'double'),
 ('V14', 'double'),
 ('V15', 'double'),
 ('V16', 'double'),
 ('V17', 'double'),
 ('V18', 'double'),
 ('V19', 'double'),
 ('V20', 'double'),
 ('V21', 'double'),
 ('V22', 'double'),
 ('V23', 'double'),
 ('V24', 'double'),
 ('V25', 'double'),
 ('V26', 'double'),
 ('V27', 'double'),
 ('V28', 'double'),
 ('Amount', 'double'),
 ('Class', 'int')]

In [10]:
df.count()

284807

In [11]:
df.rdd.getNumPartitions()

24

In [12]:
spark.sparkContext.defaultParallelism

24

In [13]:
import os

print(os.cpu_count())

24


In [14]:
df.show(5, truncate=False)

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|V1                |V2                 |V3              |V4                |V5                 |V6                 |V7                 |V8                |V9                |V10                |V11               |V12               |V13               |V14               |V15               |V16               |V17               |V18                |V19               |V20                |V21                 |V22                |V23  

In [15]:
df.groupBy("Class").count().show()

+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [16]:
total = df.count()

frauds = (
    df.filter(df.Class == 1)
      .count()
)

print(f"Total transactions : {total:,}")
print(f"Fraud transactions : {frauds:,}")
print(f"Fraud percentage   : {frauds / total * 100:.4f}%")

Total transactions : 284,807
Fraud transactions : 492
Fraud percentage   : 0.1727%


In [17]:
print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

Rows    : 284,807
Columns : 31


In [18]:
from pyspark.sql.functions import col, sum, when

nulls = df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

nulls.show(truncate=False)

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time|V1 |V2 |V3 |V4 |V5 |V6 |V7 |V8 |V9 |V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|0   |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0     |0    |
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [19]:
df.describe().show()

+-------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|summary|             Time|                  V1|                  V2|                  V3|                  V4|                  V5|                  V6|                  V7|                  V8|                  V9|                 V10|                V11|                 V12|                 V13|                 V14|                 V15|     

In [20]:
df.groupBy("Class").count().orderBy("Class").show()

+-----+------+
|Class| count|
+-----+------+
|    0|284315|
|    1|   492|
+-----+------+



In [21]:
df.select("Amount").describe().show()

+-------+-----------------+
|summary|           Amount|
+-------+-----------------+
|  count|           284807|
|   mean|88.34961925093017|
| stddev|250.1201092401885|
|    min|              0.0|
|    max|         25691.16|
+-------+-----------------+



In [22]:
print(f"Partitions: {df.rdd.getNumPartitions()}")

Partitions: 24


In [24]:
df.explain()

== Physical Plan ==
FileScan csv [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/thoma/Proyectos/fraud-detection-ai/data/raw/creditcard...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Time:double,V1:double,V2:double,V3:double,V4:double,V5:double,V6:double,V7:double,V8:doubl...




In [26]:
spark.stop()

To continue we have to start a new SparkSession

In [5]:
df_amount = df.select("Amount", "Class")

In [6]:
df_amount.show(5, truncate=False)

+------+-----+
|Amount|Class|
+------+-----+
|149.62|0    |
|2.69  |0    |
|378.66|0    |
|123.5 |0    |
|69.99 |0    |
+------+-----+
only showing top 5 rows


In [7]:
df_amount.explain()

== Physical Plan ==
FileScan csv [Amount#46,Class#47] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/thoma/Proyectos/fraud-detection-ai/data/raw/creditcard...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Amount:double,Class:int>




In [8]:
fraud_df = (
    df
    .filter(df.Class == 1)
    .select("Time", "Amount", "Class")
)

In [9]:
fraud_df.show(10)

+------+------+-----+
|  Time|Amount|Class|
+------+------+-----+
| 406.0|   0.0|    1|
| 472.0| 529.0|    1|
|4462.0|239.93|    1|
|6986.0|  59.0|    1|
|7519.0|   1.0|    1|
|7526.0|   1.0|    1|
|7535.0|   1.0|    1|
|7543.0|   1.0|    1|
|7551.0|   1.0|    1|
|7610.0|   1.0|    1|
+------+------+-----+
only showing top 10 rows


In [10]:
fraud_df.explain()

== Physical Plan ==
*(1) Filter (isnotnull(Class#47) AND (Class#47 = 1))
+- FileScan csv [Time#17,Amount#46,Class#47] Batched: false, DataFilters: [isnotnull(Class#47), (Class#47 = 1)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/thoma/Proyectos/fraud-detection-ai/data/raw/creditcard...., PartitionFilters: [], PushedFilters: [IsNotNull(Class), EqualTo(Class,1)], ReadSchema: struct<Time:double,Amount:double,Class:int>




In [15]:
fraud_df = (
    df
    .filter(df.Class == 1)
    .filter(df.Amount > 100)
    .select("Time", "Amount")
)

In [16]:
df_amount.explain()

== Physical Plan ==
FileScan csv [Amount#46,Class#47] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/thoma/Proyectos/fraud-detection-ai/data/raw/creditcard...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Amount:double,Class:int>




In [17]:
print(fraud_df)

DataFrame[Time: double, Amount: double]


In [18]:
fraud_df.show(5)

+-------+-------+
|   Time| Amount|
+-------+-------+
|  472.0|  529.0|
| 4462.0| 239.93|
| 9064.0|1809.68|
|12393.0| 179.66|
|17838.0| 766.36|
+-------+-------+
only showing top 5 rows


In [19]:
fraud_df.count()

130

In [20]:
fraud_df.explain("extended")

== Parsed Logical Plan ==
'Project ['Time, 'Amount]
+- Filter (Amount#46 > cast(100 as double))
   +- Filter (Class#47 = 1)
      +- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Analyzed Logical Plan ==
Time: double, Amount: double
Project [Time#17, Amount#46]
+- Filter (Amount#46 > cast(100 as double))
   +- Filter (Class#47 = 1)
      +- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Optimized Logical Plan ==
Project [Time#17, Amount#46]
+- Filter ((isnotnull(Class#47) AND isnotnull(Amount#46)) AND ((Class#47 = 1) AND (Amount#46 > 100.0)))
   +- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V1

In [21]:
from pyspark.sql.functions import round

df_amount_round = (
    df
    .withColumn(
        "Amount_Rounded",
        round("Amount", 0)
    )
)

In [22]:
df_amount_round.select(
    "Amount",
    "Amount_Rounded"
).show(10)

+------+--------------+
|Amount|Amount_Rounded|
+------+--------------+
|149.62|         150.0|
|  2.69|           3.0|
|378.66|         379.0|
| 123.5|         124.0|
| 69.99|          70.0|
|  3.67|           4.0|
|  4.99|           5.0|
|  40.8|          41.0|
|  93.2|          93.0|
|  3.68|           4.0|
+------+--------------+
only showing top 10 rows


In [23]:
df_amount_round.explain("extended")

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(Amount_Rounded, 'round('Amount, 0), None)]
+- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Analyzed Logical Plan ==
Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, ... 7 more fields
Project [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#37, V21#38, V22#39, V23#40, V24#41, ... 7 more fields]
+- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#3

In [24]:
df_amount_round.explain(mode="formatted")

== Physical Plan ==
* Project (2)
+- Scan csv  (1)


(1) Scan csv 
Output [31]: [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#37, V21#38, V22#39, V23#40, V24#41, V25#42, V26#43, V27#44, V28#45, Amount#46, Class#47]
Batched: false
Location: InMemoryFileIndex [file:/c:/Users/thoma/Proyectos/fraud-detection-ai/data/raw/creditcard.csv]
ReadSchema: struct<Time:double,V1:double,V2:double,V3:double,V4:double,V5:double,V6:double,V7:double,V8:double,V9:double,V10:double,V11:double,V12:double,V13:double,V14:double,V15:double,V16:double,V17:double,V18:double,V19:double,V20:double,V21:double,V22:double,V23:double,V24:double,V25:double,V26:double,V27:double,V28:double,Amount:double,Class:int>

(2) Project [codegen id : 1]
Output [32]: [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#

In [25]:
from pyspark.sql.functions import when

In [26]:
df_amount_category = (
    df
    .withColumn(
        "Amount_Category",
        when(df.Amount < 50, "Low")
        .when(df.Amount <= 200, "Medium")
        .otherwise("High")
    )
)

In [28]:
df_amount_category.select("Time", "Amount", "Amount_Category").show(10)

+----+------+---------------+
|Time|Amount|Amount_Category|
+----+------+---------------+
| 0.0|149.62|         Medium|
| 0.0|  2.69|            Low|
| 1.0|378.66|           High|
| 1.0| 123.5|         Medium|
| 2.0| 69.99|         Medium|
| 2.0|  3.67|            Low|
| 4.0|  4.99|            Low|
| 7.0|  40.8|            Low|
| 7.0|  93.2|         Medium|
| 9.0|  3.68|            Low|
+----+------+---------------+
only showing top 10 rows


In [29]:
from pyspark.sql.functions import col

In [30]:
df_amount_double = (
    df
    .withColumn(
        "Double_Amount",
        col("Amount") * 2
    )
)

In [31]:
df_amount_double.select("Time", "Amount", "Double_Amount").show(10)

+----+------+-------------+
|Time|Amount|Double_Amount|
+----+------+-------------+
| 0.0|149.62|       299.24|
| 0.0|  2.69|         5.38|
| 1.0|378.66|       757.32|
| 1.0| 123.5|        247.0|
| 2.0| 69.99|       139.98|
| 2.0|  3.67|         7.34|
| 4.0|  4.99|         9.98|
| 7.0|  40.8|         81.6|
| 7.0|  93.2|        186.4|
| 9.0|  3.68|         7.36|
+----+------+-------------+
only showing top 10 rows


In [32]:
df_amount_double.explain("extended")

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(Double_Amount, '`*`('Amount, 2), None)]
+- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Analyzed Logical Plan ==
Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, ... 7 more fields
Project [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#37, V21#38, V22#39, V23#40, V24#41, ... 7 more fields]
+- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V

In [33]:
df_class_count = (
    df
    .groupBy("Class")
    .count()
)

In [34]:
df_class_count.show()

+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [35]:
df_class_count.explain("extended")

== Parsed Logical Plan ==
'Aggregate ['Class], ['Class, 'count(1) AS count#252]
+- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Analyzed Logical Plan ==
Class: int, count: bigint
Aggregate [Class#47], [Class#47, count(1) AS count#252L]
+- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Optimized Logical Plan ==
Aggregate [Class#47], [Class#47, count(1) AS count#252L]
+- Project [Class#47]
   +- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Class#4

In [4]:
class_info = spark.createDataFrame(
    [
        (0, "Normal Transaction"),
        (1, "Fraud")
    ],
    ["Class", "Description"]
)

In [5]:
class_info.show()

+-----+------------------+
|Class|       Description|
+-----+------------------+
|    0|Normal Transaction|
|    1|             Fraud|
+-----+------------------+



In [8]:
df.show(5, truncate=False)  

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|V1                |V2                 |V3              |V4                |V5                 |V6                 |V7                 |V8                |V9                |V10                |V11               |V12               |V13               |V14               |V15               |V16               |V17               |V18                |V19               |V20                |V21                 |V22                |V23  

In [9]:
df_join = (
    df
    .join(
        class_info,
        on="Class",
        how="inner"
    )
)

In [10]:
df_join.show(5, truncate=False)

+-----+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+------------------+
|Class|Time|V1                |V2                 |V3              |V4                |V5                 |V6                 |V7                 |V8                |V9                |V10                |V11               |V12               |V13               |V14               |V15               |V16               |V17               |V18                |V19               |V20                |V21                 |

In [11]:
df_join.explain("extended")

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [Class])
:- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#49,V15#50,V16#51,V17#52,V18#53,V19#54,V20#55,V21#56,V22#57,V23#58,V24#59,... 6 more fields] csv
+- LogicalRDD [Class#9L, Description#10], false

== Analyzed Logical Plan ==
Class: int, Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, ... 7 more fields
Project [Class#65, Time#35, V1#36, V2#37, V3#38, V4#39, V5#40, V6#41, V7#42, V8#43, V9#44, V10#45, V11#46, V12#47, V13#48, V14#49, V15#50, V16#51, V17#52, V18#53, V19#54, V20#55, V21#56, V22#57, V23#58, ... 7 more fields]
+- Join Inner, (cast(Class#65 as bigint) = Class#9L)
   :- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40

Vemos en el plan de ejecución que no se ha hecho Broadcast al no detectar el pequeño tamaño de la tabla con la que se hace el join. Vamos a forzarlo en el siguiente ejemplo.

In [12]:
from pyspark.sql.functions import broadcast

In [13]:
df_join_broadcast = (
    df
    .join(
        broadcast(class_info),
        on="Class",
        how="inner"
    )
)

In [14]:
df_join_broadcast.explain("extended")

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [Class])
:- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#49,V15#50,V16#51,V17#52,V18#53,V19#54,V20#55,V21#56,V22#57,V23#58,V24#59,... 6 more fields] csv
+- ResolvedHint (strategy=broadcast)
   +- LogicalRDD [Class#9L, Description#10], false

== Analyzed Logical Plan ==
Class: int, Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, ... 7 more fields
Project [Class#65, Time#35, V1#36, V2#37, V3#38, V4#39, V5#40, V6#41, V7#42, V8#43, V9#44, V10#45, V11#46, V12#47, V13#48, V14#49, V15#50, V16#51, V17#52, V18#53, V19#54, V20#55, V21#56, V22#57, V23#58, ... 7 more fields]
+- Join Inner, (cast(Class#65 as bigint) = Class#9L)
   :- Relatio

In [15]:
df.rdd.getNumPartitions()

24

In [16]:
df.repartition(10)

DataFrame[Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, V25: double, V26: double, V27: double, V28: double, Amount: double, Class: int]

In [17]:
df_10 = df.repartition(10)

df_10.rdd.getNumPartitions()

10

In [18]:
df_10.explain("extended")

== Parsed Logical Plan ==
Repartition 10, true
+- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#49,V15#50,V16#51,V17#52,V18#53,V19#54,V20#55,V21#56,V22#57,V23#58,V24#59,... 6 more fields] csv

== Analyzed Logical Plan ==
Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, ... 6 more fields
Repartition 10, true
+- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#49,V15#50,V16#51,V17#52,V18#53,V19#54,V20#55,V21#56,V22#57,V23#58,V24#59,... 6 more fields] csv

== Optimized Logical Plan ==
Repartition 10, true
+- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#49,V

In [19]:
df_coalesce = df.coalesce(10)

In [20]:
df_coalesce.rdd.getNumPartitions()

10

In [21]:
df_coalesce.explain("extended")

== Parsed Logical Plan ==
Repartition 10, false
+- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#49,V15#50,V16#51,V17#52,V18#53,V19#54,V20#55,V21#56,V22#57,V23#58,V24#59,... 6 more fields] csv

== Analyzed Logical Plan ==
Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, ... 6 more fields
Repartition 10, false
+- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#49,V15#50,V16#51,V17#52,V18#53,V19#54,V20#55,V21#56,V22#57,V23#58,V24#59,... 6 more fields] csv

== Optimized Logical Plan ==
Repartition 10, false
+- Relation [Time#35,V1#36,V2#37,V3#38,V4#39,V5#40,V6#41,V7#42,V8#43,V9#44,V10#45,V11#46,V12#47,V13#48,V14#4

In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

In [5]:
window_spec = Window.partitionBy("Class")

In [8]:
df_avg = (
    df
    .withColumn(
        "Avg_Class_Amount",
        avg("Amount").over(window_spec)
    )
)

In [9]:
df_avg.select("Class", "Amount", "Avg_Class_Amount").show(10)

+-----+------+-----------------+
|Class|Amount| Avg_Class_Amount|
+-----+------+-----------------+
|    0|149.62|88.29102242225574|
|    0|  2.69|88.29102242225574|
|    0|378.66|88.29102242225574|
|    0| 123.5|88.29102242225574|
|    0| 69.99|88.29102242225574|
|    0|  3.67|88.29102242225574|
|    0|  4.99|88.29102242225574|
|    0|  40.8|88.29102242225574|
|    0|  93.2|88.29102242225574|
|    0|  3.68|88.29102242225574|
+-----+------+-----------------+
only showing top 10 rows


In [10]:
df_avg.explain("extended")

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(Avg_Class_Amount, 'avg('Amount) windowspecdefinition('Class, unspecifiedframe$()), None)]
+- Relation [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] csv

== Analyzed Logical Plan ==
Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, ... 7 more fields
Project [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#37, V21#38, V22#39, V23#40, V24#41, ... 7 more fields]
+- Project [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23

In [13]:
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

In [14]:
window_rank = (
    Window
    .partitionBy("Class")
    .orderBy(col("Amount").desc())
)

In [15]:
df_rank = (
    df
    .withColumn(
        "row_number",
        row_number().over(window_rank)
    )
)

In [16]:
df_rank.select(
    "Class",
    "Amount",
    "row_number"
).show(20)

+-----+--------+----------+
|Class|  Amount|row_number|
+-----+--------+----------+
|    0|25691.16|         1|
|    0|19656.53|         2|
|    0| 18910.0|         3|
|    0|12910.93|         4|
|    0|11898.09|         5|
|    0|11789.84|         6|
|    0|10199.44|         7|
|    0| 10000.0|         8|
|    0| 8790.26|         9|
|    0|  8787.0|        10|
|    0|  8360.0|        11|
|    0|  8182.7|        12|
|    0| 7879.42|        13|
|    0| 7862.39|        14|
|    0|  7766.6|        15|
|    0| 7712.43|        16|
|    0|  7636.3|        17|
|    0| 7583.32|        18|
|    0|  7541.7|        19|
|    0| 7429.15|        20|
+-----+--------+----------+
only showing top 20 rows


In [17]:
top_transactions = (
    df_rank
    .filter(col("row_number") == 1)
)

top_transactions.show()

+--------+-----------------+-----------------+-----------------+----------------+-----------------+-------------------+----------------+------------------+-----------------+-----------------+-----------------+------------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+------------------+----------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+-----------------+--------+-----+----------+
|    Time|               V1|               V2|               V3|              V4|               V5|                 V6|              V7|                V8|               V9|              V10|              V11|               V12|              V13|              V14|              V15|               V16|               V17|              V18|               V19|             V20|              V21|              V22|              V23|              V24|       

In [18]:
top_transactions.explain("extended")

== Parsed Logical Plan ==
'Filter '`=`('row_number, 1)
+- Project [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#37, V21#38, V22#39, V23#40, V24#41, ... 7 more fields]
   +- Project [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#37, V21#38, V22#39, V23#40, V24#41, ... 8 more fields]
      +- Window [row_number() windowspecdefinition(Class#47, Amount#46 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS row_number#65], [Class#47], [Amount#46 DESC NULLS LAST]
         +- Project [Time#17, V1#18, V2#19, V3#20, V4#21, V5#22, V6#23, V7#24, V8#25, V9#26, V10#27, V11#28, V12#29, V13#30, V14#31, V15#32, V16#33, V17#34, V18#35, V19#36, V20#37, V21#38, V22#39, V23#40, V24#41, ... 6 more fields]
            +- Relation [Time#17,V1#18,V2#19,V3